# Run chính — train, chấm test ở mọi snapshot, đẩy kết quả
Một run của Giai đoạn 3 (PLAN §5.1, EXPERIMENTS.md "Khóa giao thức"): `CONFIG` là `e1_seq`, `e2_mix`, `e3_lora_r64` hoặc `e3_lora_r8`, `SEED` là 0, 1 hoặc 2. Cell 6 làm hết một mạch: train (hoặc resume), chấm **test** ở mọi snapshot, perplexity và KL của snapshot cuối, rồi đẩy kết quả. Dùng runtime **T4 GPU**; một run full fine-tune khoảng 6 giờ GPU, nên phải chạy qua nhiều lần.

Trên Hub:
- Repo model `<username>/smollm2-360m-<run>` (public với run chính): snapshot, hai ô `checkpoint-a/`, `checkpoint-b/` để resume (luân phiên: còn checkpoint trước đó nếu bản mới nhất hỏng; lịch sử được squash sau mỗi lần đẩy, xóa khi run xong), metric train.
- Repo private `<username>/smollm2-360m-results`, thư mục `<run>/`: output và metric eval, đẩy từng snapshot ngay khi chấm xong. Output chứa nguyên văn bộ test (rút từ dữ liệu gated) nên không bao giờ để public.

Bị ngắt ở bất kỳ đâu: kết nối lại, chạy lại cell 1–5 rồi cell 6. Cell 6 tự resume từ checkpoint mới nhất dùng được nếu train chưa xong, bỏ qua train nếu đã xong, và bỏ qua snapshot đã có kết quả trong repo kết quả.


## 1. Cài thư viện

In [ ]:
%pip uninstall -q -y torchao
%pip install -q transformers==5.20.2 huggingface-hub==1.32.0 psutil==7.2.2 safetensors==0.8.0 pyyaml==6.0.3 accelerate==1.15.0 datasets==5.1.0 peft==0.21.2 trl==1.16.0
from importlib.metadata import PackageNotFoundError, version
wanted = {"transformers": "5.20.2", "huggingface-hub": "1.32.0", "psutil": "7.2.2", "safetensors": "0.8.0", "pyyaml": "6.0.3", "accelerate": "1.15.0", "datasets": "5.1.0", "peft": "0.21.2", "trl": "1.16.0"}
installed = {name: version(name) for name in wanted}
print(installed)
assert installed == wanted, "Phiên bản chưa khớp requirements.txt; chạy lại cell này."
try:
    raise AssertionError(f"torchao {version('torchao')} vẫn còn; chạy lại cell này.")
except PackageNotFoundError:
    pass


## 2. Nạp mã nguồn của dự án

In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path

payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "573a986eaa3e11cb814ef1a09a00beec72b1c1d2e6e3aceb3249c045e795cf38"
root = Path("/content/sft-base-playground")
root.mkdir(parents=True, exist_ok=True)
for relative, source in json.loads(payload).items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(root)
sys.path.insert(0, str(root))
# One GPU only (Kaggle gives two T4s): with two, Trainer splits each batch across them and the effective batch doubles.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Các object đã tạo (như lab) vẫn giữ nguyên; lần import tiếp dùng source mới.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Source bundle SHA-256:", "573a986eaa3e11cb814ef1a09a00beec72b1c1d2e6e3aceb3249c045e795cf38")
print("Working directory:", root)


## 3. Kiểm tra mã

In [ ]:
import subprocess
checks = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr)
checks.check_returncode()


## 4. Đăng nhập HF Hub

Cần secret `HF_TOKEN` trong 🔑 **Secrets** của Colab, đã bật **Notebook access**. Chạy cell này **bằng nút ▶ trên giao diện Colab**: secret không đọc được khi cell được chạy qua MCP.

Chạy trên máy Kaggle (Colab nối vào Kaggle Jupyter Server) thì secret của Colab có thể không đọc được: khi đó cell này đọc secret `HF_TOKEN` trong **Add-ons → Secrets** của notebook Kaggle.

Điền `DATASET_REPO` (dataset private chứa `eval_sets/` và `train/`).

In [ ]:
from huggingface_hub import whoami


def hf_token():
    """The HF_TOKEN secret of Colab, or of Kaggle when this runtime is a Kaggle Jupyter Server."""
    try:
        from google.colab import userdata

        return userdata.get("HF_TOKEN")
    except Exception as error:  # Colab's secret store may be unreachable from a Kaggle backend
        print("Colab secret unavailable:", type(error).__name__, error)
    from kaggle_secrets import UserSecretsClient

    return UserSecretsClient().get_secret("HF_TOKEN")


os.environ["HF_TOKEN"] = hf_token()
DATASET_REPO = ""  # ví dụ "your-username/sft-capability-data"
HF_USER = whoami()["name"]
RESULTS_REPO = f"{HF_USER}/smollm2-360m-results"  # private: outputs quote the gated eval sets
print("Logged in as:", HF_USER)
assert DATASET_REPO, "Điền DATASET_REPO trước khi chạy tiếp."


## 5. Tải dev/test, D_chat/D_tool và kiểm hash
Dừng nếu `src.eval.manifest` báo file thiếu hoặc khác; hash của D_chat/D_tool phải khớp `data/TRAIN.md`.

In [ ]:
import hashlib
from src.eval.run_eval import fetch
from src.train import fetch_train_sets
fetch(DATASET_REPO)
fetch_train_sets(DATASET_REPO, Path("data/train"))
!python -m src.eval.manifest
for name in ("d_chat", "d_tool"):
    print(name, hashlib.sha256(Path(f"data/train/{name}.jsonl").read_bytes()).hexdigest())


## 6. Train, chấm test, đẩy kết quả
Chọn `CONFIG` và `SEED`. Ghi giả thuyết của run vào EXPERIMENTS.md **trước** lần chạy đầu (PLAN Giai đoạn 3). Theo dõi thay vì chờ cell trả lời: lúc train, đọc `progress.json` của repo model (step, token, snapshot gần nhất, giờ đẩy; cập nhật mỗi checkpoint); lúc chấm, mỗi snapshot xong là một commit trên repo kết quả; commit cuối của cell là `<run>: knowledge`.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download, upload_file
from huggingface_hub.utils import EntryNotFoundError, RepositoryNotFoundError
from src.train import load_config

CONFIG, SEED = "e2_mix", 0
RUN = load_config(f"configs/{CONFIG}.yaml", seed=SEED)["run_name"]
HUB_MODEL_ID = f"{HF_USER}/smollm2-360m-{RUN.replace('_', '-')}"
api = HfApi()


def train_state():
    """done, resume or new, from what the run repo holds."""
    try:
        files = set(api.list_repo_files(HUB_MODEL_ID))
    except RepositoryNotFoundError:
        return "new"
    if "metrics.json" in files and json.loads(Path(hf_hub_download(HUB_MODEL_ID, "metrics.json")).read_text())["completed"]:
        return "done"
    return "resume" if "resume.json" in files else "new"


state = train_state()
print(RUN, HUB_MODEL_ID, "train:", state)
if state != "done":
    resume = "--resume hub" if state == "resume" else ""
    !python -m src.train --config configs/{CONFIG}.yaml --seed {SEED} --hub-model-id {HUB_MODEL_ID} --data-repo {DATASET_REPO} {resume}
    assert train_state() == "done", "Train chưa xong (xem log ở trên); chạy lại cell này để resume."
!python -m src.eval.run_checkpoints --run {RUN} --hub-model-id {HUB_MODEL_ID} --split test --results-repo {RESULTS_REPO} --cleanup
snapshots = sorted(path for path in api.list_repo_files(HUB_MODEL_ID) if path.startswith("snapshots/") and path.endswith("/meta.json"))
final = snapshots[-1].split("/")[1]
local = Path(f"results/runs/{RUN}/snapshots/{final}")
if not local.exists():
    api.snapshot_download(HUB_MODEL_ID, allow_patterns=[f"snapshots/{final}/*"], local_dir=f"results/runs/{RUN}")
!python -m src.eval.knowledge --model {local} --device cuda --output results/{RUN}/knowledge-{final}.json
upload_file(path_or_fileobj=f"results/{RUN}/knowledge-{final}.json", path_in_repo=f"{RUN}/knowledge-{final}.json",
            repo_id=RESULTS_REPO, commit_message=f"{RUN}: knowledge")
print("done", RUN)
